In [ ]:
import random

from qibo import (
    Circuit, 
    gates, 
    hamiltonians,
    symbols,
    set_backend,
)

from mpstab.models.ansatze import TranspiledAnsatz, HardwareEfficient
from mpstab.evolutors.hsmpo import HSMPO

set_backend("qibojit", platform="numba")

nqubits = 5
nlayers = 3
obs_str = "Y" * nqubits

def construct_circuit(nqubits: int, nlayers: int):
    """Conctruct a target quantum circuit."""
    c = Circuit(nqubits)
    for _ in range(nlayers):
        for q in range(nqubits):
            c.add(gates.RZ(q=q, theta=random.random()))
        [c.add(gates.CNOT(q0=q%nqubits, q1=(q+1)%nqubits)) for q in range(nqubits)]
    return c

def initial_state(nqubits: int):
    """Construct a random initial state."""
    c = Circuit(nqubits)
    for q in range(nqubits):
        c.add(gates.RY(q=q, theta=random.random()))
    return c

print("Target circuit:")
circ = construct_circuit(nqubits=nqubits, nlayers=nlayers)
circ.draw()

print("\nInitial state circuit:")
init_psi = initial_state(nqubits=nqubits)
init_psi.draw()

circ = construct_circuit(nqubits=nqubits, nlayers=nlayers)

# Construct a transpiled ansatz given a circuit
# Arguments here are native gates and chip topology 
ans = TranspiledAnsatz(original_circuit=circ)

# Construct the Hybrid Surrogate
hs = HSMPO(ansatz=ans, initial_state=init_psi) # Here one can add an initial state (a circuit)

[Qibo 0.2.19|INFO|2025-05-07 09:20:27]: Using qibojit (numba) backend on /CPU:0


Target circuit:
0: ─RZ─o───────X─RZ─o───────X─RZ─o───────X─
1: ─RZ─X─o─────|─RZ─X─o─────|─RZ─X─o─────|─
2: ─RZ───X─o───|─RZ───X─o───|─RZ───X─o───|─
3: ─RZ─────X─o─|─RZ─────X─o─|─RZ─────X─o─|─
4: ─RZ───────X─o─RZ───────X─o─RZ───────X─o─

Initial state circuit:
0: ─RY─
1: ─RY─
2: ─RY─
3: ─RY─
4: ─RY─


In [12]:
full_circuit, magic_layers, stab_layers = ans.partitionate_circuit(
    replacement_probability=0.5
)

# Original circuit
circ.draw()

0: ─RZ─o───────X─RZ─o───────X─RZ─o───────X─
1: ─RZ─X─o─────|─RZ─X─o─────|─RZ─X─o─────|─
2: ─RZ───X─o───|─RZ───X─o───|─RZ───X─o───|─
3: ─RZ─────X─o─|─RZ─────X─o─|─RZ─────X─o─|─
4: ─RZ───────X─o─RZ───────X─o─RZ───────X─o─


In [13]:
# Transpiled and quasi-cliffordized circuit
full_circuit.draw()

0:     ─RZ────────o────────────────────────────Z─GPI2─Z─Z──GPI2─RZ───o─────── ...
1:     ─RZ─Z─GPI2─Z─Z─GPI2─o──────────────────────────|─RZ─Z────GPI2─Z─Z─GPI2 ...
2:     ─RZ──────────Z─GPI2─Z─Z─GPI2─o─────────────────|─RZ─────────────Z─GPI2 ...
3:     ─RZ───────────────────Z─GPI2─Z─Z─GPI2─o────────|─RZ─────────────────── ...
4:     ─RZ────────────────────────────Z─GPI2─Z─Z─GPI2─o─RZ─────────────────── ...

0: ... ─────────────────────Z─GPI2─Z─Z──GPI2─RZ───o────────────────────────── ...
1: ... ─o──────────────────────────|─RZ─Z────GPI2─Z─Z─GPI2─o───────────────── ...
2: ... ─Z─Z─GPI2─o─────────────────|─RZ─────────────Z─GPI2─Z─Z─GPI2─o──────── ...
3: ... ───Z─GPI2─Z─Z─GPI2─o────────|─RZ──────────────────────Z─GPI2─Z─Z─GPI2─ ...
4: ... ────────────Z─GPI2─Z─Z─GPI2─o─RZ───────────────────────────────Z─GPI2─ ...

0: ... ──Z─GPI2─Z─Z─GPI2─
1: ... ─────────|────────
2: ... ─────────|────────
3: ... o────────|────────
4: ... Z─Z─GPI2─o────────


In [38]:
# Sample a similar circuit and compute expval
expval, partitions = hs.expectation_from_partition(
    observable=obs_str,
    return_partitions=True,
    replacement_probability=0.7,
)

expval

4.835602169630727e-16

In [36]:
# Construct Hamiltonian.
form = 1
for i, pauli in enumerate(obs_str):
    form *= getattr(symbols, pauli)(i)
ham = hamiltonians.SymbolicHamiltonian(form=form)

In [37]:
ham.expectation((init_psi+partitions["full_circuit"])().state())

np.float64(1.0408340855860843e-16)

### Construct a target quantum circuit

Target circuit:
0: ─RZ─o───X─RZ─o───X─
1: ─RZ─X─o─|─RZ─X─o─|─
2: ─RZ───X─o─RZ───X─o─

Initial state circuit:
0: ─RY─
1: ─RY─
2: ─RY─


### Transpiled ansatz

In [7]:
ans.circuit.draw()

0:     ─RZ────────o──────────Z─GPI2─Z─Z──GPI2─RZ───o──────────Z─GPI2─Z─Z─GPI2 ...
1:     ─RZ─Z─GPI2─Z─Z─GPI2─o────────|─RZ─Z────GPI2─Z─Z─GPI2─o────────|─────── ...
2:     ─RZ──────────Z─GPI2─Z─Z─GPI2─o─RZ─────────────Z─GPI2─Z─Z─GPI2─o─────── ...

0: ... ─
1: ... ─
2: ... ─


### Hybrid surrogate

In [ ]:

hdw_hs = HSMPO(
    ansatz=hdw_ans,
    initial_state=init_psi,
)

In [13]:
expval, partitions = hs.expectation_from_partition(
    observable=obs_str,
    return_partitions=True,
    replacement_probability=0.6,
)

expval

<qibo.gates.gates.RZ object at 0x7f522998e890> (0.41788205833524417,)
<qibo.gates.gates.RZ object at 0x7f522977c290> (0.034508046418373906,)
<qibo.gates.gates.RZ object at 0x7f522977d990> (0.7073553489034012,)
<qibo.gates.gates.RZ object at 0x7f522a1f7fd0> (0.5467568866725483,)
<qibo.gates.gates.RZ object at 0x7f522976c710> (0.965917494495791,)
<qibo.gates.gates.RZ object at 0x7f522976c590> (0.5484299537564308,)
0: ─RZ─
1: ────
2: ────
0: ───────────o──────────Z─GPI2─Z─Z─GPI2─
1: ─RZ─Z─GPI2─Z─Z─GPI2─o────────|────────
2: ─RZ──────────Z─GPI2─Z─Z─GPI2─o────────
0: ────────o──────────Z─GPI2─Z─Z─GPI2─
1: ─Z─GPI2─Z─Z─GPI2─o────────|────────
2: ──────────Z─GPI2─Z─Z─GPI2─o────────


0.8092304992113187

In [ ]:
hs = HSMPO(
    ansatz=ans,
    initial_state=init_psi
)

expval, partitions = hs.expectation_from_partition(
    replacement_probability=0.6,
    observable=obs_str,
    return_partitions=True,
)

expval

0.5502722890060072

In [26]:
print(
    ham.expectation((init_psi+circuit)().state())
)
print(
    ham.expectation((init_psi+circ)().state())
)

0.21638583864243272
0.2163858386424341
